In [ ]:

# koz3-tpu-probe-v3 — hardware probe (notebook)
import json, os, platform, shutil, subprocess, sys, time
from pathlib import Path
T0=time.time()
info={"hostname":platform.node(),"python":sys.version,"platform":platform.platform(),"nproc":os.cpu_count()}
try:
    mem=Path("/proc/meminfo").read_text().splitlines()
    info["meminfo"]={l.split(":")[0]:l.split(":")[1].strip() for l in mem if l.startswith(("MemTotal","MemAvailable"))}
except Exception as e:
    info["mem_error"]=repr(e)
try:
    info["lscpu"]=subprocess.run(["lscpu"],capture_output=True,text=True,timeout=30).stdout.strip().splitlines()[:50]
except Exception as e:
    info["lscpu_error"]=repr(e)
try:
    du={}
    for p in ("/","/kaggle/working","/tmp"):
        u=shutil.disk_usage(p); du[p]=dict(total_gb=round(u.total/2**30,2),free_gb=round(u.free/2**30,2))
    info["disk"]=du
except Exception as e:
    info["disk_error"]=repr(e)
info["env_tpu"]={k:os.environ.get(k) for k in sorted(os.environ) if any(s in k.upper() for s in ("TPU","XLA","PJRT","ISTPU","KAGGLE_TPU"))}
info["dev_accel"]=sorted(str(p) for p in Path("/dev").glob("accel*"))
for pkg in ("torch","torch_xla","jax","jaxlib","lightgbm","numpy"):
    try:
        from importlib.metadata import version
        info[f"pkg_{pkg}"]=version(pkg)
    except Exception:
        info[f"pkg_{pkg}"]=None
# torch_xla smoke
try:
    import torch, torch_xla, torch_xla.runtime as xr
    d=torch_xla.device()
    v=float(((torch.ones(8,device=d)*2).sum()).cpu())
    info["xla"]=dict(ok=(v==16.0),v=v,device_type=xr.device_type(),n_dev=len(torch_xla.devices()),accel=os.environ.get("TPU_ACCELERATOR_TYPE"))
except Exception as e:
    info["xla_error"]=repr(e)
# lightgbm CPU smoke on many cores
try:
    import lightgbm as lgb, numpy as np
    X=np.random.randn(2000,20); y=(X[:,0]>0).astype(int)
    m=lgb.LGBMClassifier(n_estimators=20,n_jobs=-1,verbose=-1); m.fit(X,y)
    info["lgbm_ok"]=True; info["lgbm_n_jobs"]=m.n_jobs
except Exception as e:
    info["lgbm_error"]=repr(e)
info["elapsed_s"]=round(time.time()-T0,2)
Path("/kaggle/working/tpu_probe.json").write_text(json.dumps(info,indent=2))
print(json.dumps(info,indent=2)[:3000])
print("WROTE tpu_probe.json")
